# HeatSense | Phase 1: Notebook 00 — Dataset Inventory

**Objective**: Discover and catalog the physical files, directory structure, metadata fields, and basic record layouts of the Phase 1 battery dataset without making unverified assumptions or modifying source files.

**Operating Principle**: Read-only inspection. Never overwrite, move, or modify raw dataset files.

### Cell 1: Environment & Dataset Path Resolution
* **Analytical Question**: Where is the dataset physically located on this machine, and can Python access it read-only?
* **Why It Matters to HeatSense**: Ensures that the data pipeline connects to the actual data directory without hardcoding rigid machine-specific paths.
* **Expected Inputs**: Configurable environment variable `HEATSENSE_DATASET_ROOT` or fallback paths.
* **What the Code Does**: Checks `data/raw/cleaned_dataset` inside the repository and `D:/Professional Projects/Digital_twin_project/Datasets/cleaned_dataset` externally, verifying read access.
* **Output to Inspect**: Verified dataset root directory path and confirmation of access.
* **Decision**: Confirms the active data source directory for all subsequent cells.

In [1]:
import os
from pathlib import Path

# Check paths in order of priority:
candidate_paths = [
    Path(os.environ.get("HEATSENSE_DATASET_ROOT", "")) if os.environ.get("HEATSENSE_DATASET_ROOT") else None,
    Path("../../data/raw/cleaned_dataset"),
    Path("../data/raw/cleaned_dataset"),
    Path("data/raw/cleaned_dataset"),
    Path("D:/Professional Projects/Digital_twin_project/Datasets/cleaned_dataset"),
    Path("D:/Professional Projects/Digital_twin_project/heatsense-battery-digital-twin/data/raw/cleaned_dataset"),
]

DATASET_ROOT = None
for p in candidate_paths:
    if p and p.exists() and (p / "metadata.csv").exists():
        DATASET_ROOT = p.resolve()
        break

if DATASET_ROOT is None:
    raise FileNotFoundError("Dataset root with metadata.csv could not be found. Please set HEATSENSE_DATASET_ROOT.")

print(f"[SUCCESS] Active Dataset Root: {DATASET_ROOT}")
print(f"[CHECK] Exists: {DATASET_ROOT.exists()}")

[SUCCESS] Active Dataset Root: D:\Professional Projects\Digital_twin_project\heatsense-battery-digital-twin\data\raw\cleaned_dataset
[CHECK] Exists: True


### Cell 2: Top-Level Directory Manifest & Size Audit
* **Analytical Question**: What subdirectories and root files exist inside the dataset root, and what are their storage footprints?
* **Why It Matters to HeatSense**: Confirms whether expected subfolders (`data/`, `extra_infos/`, `metadata.csv`) are present and intact.
* **Expected Inputs**: `DATASET_ROOT` from Cell 1.
* **What the Code Does**: Iterates through `DATASET_ROOT`, listing subdirectories, file counts, and sizes in megabytes.
* **Output to Inspect**: Table/list of top-level items with file counts and total MB.
* **Decision**: Confirms structural completeness of the dataset distribution.

In [2]:
import pandas as pd

manifest = []
for item in sorted(DATASET_ROOT.iterdir()):
    if item.is_dir():
        sub_files = list(item.glob("*"))
        total_size = sum(f.stat().st_size for f in sub_files if f.is_file()) / (1024 * 1024)
        manifest.append({"Item": item.name, "Type": "Directory", "Count": len(sub_files), "Size_MB": round(total_size, 2)})
    else:
        size_mb = item.stat().st_size / (1024 * 1024)
        manifest.append({"Item": item.name, "Type": "File", "Count": 1, "Size_MB": round(size_mb, 2)})

manifest_df = pd.DataFrame(manifest)
print(manifest_df.to_string(index=False))

        Item      Type  Count  Size_MB
        data Directory   7565   558.04
 extra_infos Directory      9     0.02
metadata.csv      File      1     0.81


### Cell 3: Metadata Schema & Summary Statistics
* **Analytical Question**: What does `metadata.csv` contain, what are its columns, row counts, unique battery identifiers, and test types?
* **Why It Matters to HeatSense**: `metadata.csv` is the index linking sequential run files (`00001.csv`, etc.) to battery cells, test conditions, temperatures, and measured capacities.
* **Expected Inputs**: `metadata.csv` path.
* **What the Code Does**: Reads `metadata.csv` using Pandas, checks dimensions, column data types, missing value counts, and lists unique batteries and test types.
* **Output to Inspect**: Number of rows, list of columns, table of unique `battery_id` values with test counts, and breakdown of `type`.
* **Decision**: Establishes how battery cells are tracked and whether multiple test types (e.g. charge, discharge, impedance) are present.

In [3]:
meta_path = DATASET_ROOT / "metadata.csv"
meta_df = pd.read_csv(meta_path)

print(f"Metadata Dimensions: {meta_df.shape[0]} rows x {meta_df.shape[1]} columns\n")
print("Columns and Non-Null Counts:")
print(meta_df.info())
print("\nFirst 5 rows of metadata:")
display(meta_df.head())

print("\nUnique Battery IDs:", sorted(meta_df['battery_id'].dropna().unique()))
print("\nTest Type Breakdown:")
print(meta_df['type'].value_counts())
print("\nAmbient Temperature Breakdown (deg C):")
print(meta_df['ambient_temperature'].value_counts())

Metadata Dimensions: 7565 rows x 10 columns

Columns and Non-Null Counts:
<class 'pandas.DataFrame'>
RangeIndex: 7565 entries, 0 to 7564
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   type                 7565 non-null   str  
 1   start_time           7565 non-null   str  
 2   ambient_temperature  7565 non-null   int64
 3   battery_id           7565 non-null   str  
 4   test_id              7565 non-null   int64
 5   uid                  7565 non-null   int64
 6   filename             7565 non-null   str  
 7   Capacity             2794 non-null   str  
 8   Re                   1956 non-null   str  
 9   Rct                  1956 non-null   str  
dtypes: int64(3), str(7)
memory usage: 591.1 KB
None

First 5 rows of metadata:


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN



Unique Battery IDs: ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0049', 'B0050', 'B0051', 'B0052', 'B0053', 'B0054', 'B0055', 'B0056']

Test Type Breakdown:
type
charge       2815
discharge    2794
impedance    1956
Name: count, dtype: int64

Ambient Temperature Breakdown (deg C):
ambient_temperature
24    4658
4     2060
43     388
22     249
44     210
Name: count, dtype: int64


### Cell 4: Extra Infos & Documentation Audit
* **Analytical Question**: What documentation files exist in `extra_infos/` and what context do they provide regarding experimental protocols?
* **Why It Matters to HeatSense**: README files explain cell chemistry, test equipment, nominal capacity, and cutoff criteria directly from the experimental researchers.
* **Expected Inputs**: `extra_infos/` directory.
* **What the Code Does**: Lists all files in `extra_infos/` and prints the first 25 lines of a representative README file.
* **Output to Inspect**: List of text files and introductory excerpt from one NASA README.
* **Decision**: Discloses manufacturer, rated capacity, and cutoff voltage guidelines.

In [4]:
extra_dir = DATASET_ROOT / "extra_infos"
if extra_dir.exists():
    info_files = sorted(list(extra_dir.glob("*")))
    print(f"Found {len(info_files)} documentation files in extra_infos/:")
    for f in info_files:
        print(f" - {f.name} ({round(f.stat().st_size / 1024, 2)} KB)")
    
    sample_txt = info_files[0]
    print(f"\n--- Preview of {sample_txt.name} (first 25 lines) ---")
    with open(sample_txt, "r", encoding="utf-8", errors="ignore") as fp:
        for _ in range(25):
            line = fp.readline()
            if not line: break
            print(line.rstrip())
else:
    print("[WARN] extra_infos directory not found.")

Found 9 documentation files in extra_infos/:
 - README_05_06_07_18.txt (2.77 KB)
 - README_25_26_27_28.txt (2.36 KB)
 - README_29_30_31_32.txt (2.31 KB)
 - README_33_34_36.txt (2.39 KB)
 - README_38_39_40.txt (2.38 KB)
 - README_41_42_43_44.txt (2.52 KB)
 - README_45_46_47_48.txt (2.5 KB)
 - README_49_50_51_52.txt (2.53 KB)
 - README_53_54_55_56.txt (2.5 KB)

--- Preview of README_05_06_07_18.txt (first 25 lines) ---
Data Description:
A set of four Li-ion batteries (# 5, 6, 7 and 18) were run through 3 different operational profiles (charge, discharge and impedance) at room temperature. Charging was carried out in a constant current (CC) mode at 1.5A until the battery voltage reached 4.2V and then continued in a constant voltage (CV) mode until the charge current dropped to 20mA. Discharge was carried out at a constant current (CC) level of 2A until the battery voltage fell to 2.7V, 2.5V, 2.2V and 2.5V for batteries 5 6 7 and 18 respectively. Impedance measurement was carried out throu

### Cell 5: Sequential Time-Series Data File Inspection
* **Analytical Question**: How are time-series measurements formatted inside the sequential CSVs (e.g., `00001.csv`), and what are the observed sampling intervals?
* **Why It Matters to HeatSense**: Confirms exact column headers, numerical precision, sampling frequency (delta_t), and initial signal values.
* **Expected Inputs**: `data/00001.csv`.
* **What the Code Does**: Loads `00001.csv`, prints summary statistics (`describe()`), computes the distribution of time steps $\Delta t = t_k - t_{k-1}$, and checks for NaN values.
* **Output to Inspect**: Column names, data types, min/max values of voltage/current/temperature, and $\Delta t$ statistics.
* **Decision**: Determines whether sampling is uniform or variable, and verifies sensor signal boundaries.

In [5]:
sample_csv_path = DATASET_ROOT / "data" / "00001.csv"
sample_df = pd.read_csv(sample_csv_path)

print(f"Sample File: {sample_csv_path.name}")
print(f"Shape: {sample_df.shape[0]} rows x {sample_df.shape[1]} columns\n")
print("Columns:", list(sample_df.columns))
print("\nSummary Statistics:")
display(sample_df.describe())

# Compute time step delta
if "Time" in sample_df.columns and len(sample_df) > 1:
    dt_series = sample_df["Time"].diff().dropna()
    print("\nSampling Interval (dt in seconds) Summary:")
    print(f"Mean dt: {dt_series.mean():.3f}s | Median dt: {dt_series.median():.3f}s | Min dt: {dt_series.min():.3f}s | Max dt: {dt_series.max():.3f}s")

Sample File: 00001.csv
Shape: 490 rows x 6 columns

Columns: ['Voltage_measured', 'Current_measured', 'Temperature_measured', 'Current_load', 'Voltage_load', 'Time']

Summary Statistics:


,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time
count,490.000000,490.000000,490.000000,490.000000,490.000000,490.000000
mean,3.475266,-0.952767,8.272423,0.957153,2.817939,3205.134141
std,0.284626,0.201448,1.455269,0.202690,0.644985,1861.395346
min,2.470612,-1.001101,5.008084,0.000200,0.000000,0.000000
25%,3.336993,-0.995898,7.313213,1.000000,2.790500,1598.417750
50%,3.491946,-0.995207,8.184590,1.000000,2.945500,3200.492500
75%,3.655987,-0.994648,9.025544,1.000000,3.105750,4809.176250
max,4.246764,0.000252,12.376816,1.000000,4.262000,6436.141000



Sampling Interval (dt in seconds) Summary:
Mean dt: 13.162s | Median dt: 13.125s | Min dt: 9.360s | Max dt: 13.921s


### Cell 6: Linking Metadata to Sequential Data Files
* **Analytical Question**: Does row $i$ in `metadata.csv` match the file referenced in `filename` and correspond to the expected test type and duration?
* **Why It Matters to HeatSense**: Validates the indexing integrity of the dataset ingestion pipeline.
* **Expected Inputs**: `metadata.csv` row 0 and `00001.csv`.
* **What the Code Does**: Retrieves the metadata row for `00001.csv`, prints test type, battery ID, and recorded Capacity, and compares it with the final elapsed time in the data file.
* **Output to Inspect**: Consistency check table between metadata and time-series file.
* **Decision**: Confirms whether `filename` in `metadata.csv` is the canonical relational key.

In [6]:
meta_match = meta_df[meta_df["filename"] == "00001.csv"]
print("Metadata record for 00001.csv:")
print(meta_match.to_dict(orient="records"))

total_time = sample_df["Time"].max() if "Time" in sample_df.columns else None
print(f"\n00001.csv Record Count: {len(sample_df)}")
print(f"00001.csv Max Elapsed Time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Recorded Metadata Capacity: {meta_match['Capacity'].values[0]} Ah")

Metadata record for 00001.csv:
[{'type': 'discharge', 'start_time': '[2010.       7.      21.      15.       0.      35.093]', 'ambient_temperature': 4, 'battery_id': 'B0047', 'test_id': 0, 'uid': 1, 'filename': '00001.csv', 'Capacity': '1.6743047446975208', 'Re': nan, 'Rct': nan}]

00001.csv Record Count: 490
00001.csv Max Elapsed Time: 6436.14 seconds (107.27 minutes)
Recorded Metadata Capacity: 1.6743047446975208 Ah


### Cell 7: Summary of Findings & Next Action Checklist
* **Analytical Question**: What facts are verified, and what open items must be investigated in Notebook 01?
* **Why It Matters to HeatSense**: Summarizes Notebook 00 deliverables for entry into `project-planning/data_driven_decision_log.md`.
* **What the Code Does**: Aggregates the findings and prints a standardized summary checklist.
* **Output to Inspect**: Verified facts vs open items.
* **Decision**: Informs the user of exact outputs to return for review.

In [7]:
print("="*60)
print("HEATSENSE NOTEBOOK 00 EXECUTION COMPLETE")
print("="*60)
print("Please copy and share the outputs of Cells 1 to 6 with the assistant.")
print("Key items to verify:")
print(" 1. Active DATASET_ROOT path")
print(" 2. Metadata row count and unique battery IDs")
print(" 3. Breakdown of test types (discharge, charge, impedance)")
print(" 4. Sample 00001.csv column names and sampling interval statistics")
print("="*60)

HEATSENSE NOTEBOOK 00 EXECUTION COMPLETE
Please copy and share the outputs of Cells 1 to 6 with the assistant.
Key items to verify:
 1. Active DATASET_ROOT path
 2. Metadata row count and unique battery IDs
 3. Breakdown of test types (discharge, charge, impedance)
 4. Sample 00001.csv column names and sampling interval statistics
